# Combine Tables

In [1]:
import os
import numpy as np
import pandas as pd


healthy = r"featureCounts_healthy/counts.txt"
t2d = r"featureCounts_t2d/counts.txt"

In [2]:
df_healthy = pd.read_csv(healthy, sep='\t', comment="#")
df_t2d = pd.read_csv(t2d, sep='\t', comment="#")

In [3]:
expr_df_healthy = df_healthy.set_index("Geneid").iloc[:, 5:]
expr_df_t2d = df_t2d.set_index("Geneid").iloc[:, 5:]

In [4]:
print(len(expr_df_healthy.columns))
print(len(expr_df_t2d.columns))

97
45


In [24]:
expr_df_t2d.columns

Index(['HP1504101T2D_B6.bam', 'HP1504101T2D_C18.bam', 'HP1504101T2D_D9.bam',
       'HP1504101T2D_E15.bam', 'HP1504101T2D_E16.bam', 'HP1504101T2D_E24.bam',
       'HP1504101T2D_H23.bam', 'HP1504101T2D_H6.bam', 'HP1504101T2D_N10.bam',
       'HP1504101T2D_O12.bam', 'HP1508501T2D_B16.bam', 'HP1508501T2D_I8.bam',
       'HP1508501T2D_J6.bam', 'HP1526901T2D_A1.bam', 'HP1526901T2D_A14.bam',
       'HP1526901T2D_A19.bam', 'HP1526901T2D_A2.bam', 'HP1526901T2D_A5.bam',
       'HP1526901T2D_B1.bam', 'HP1526901T2D_E11.bam', 'HP1526901T2D_E2.bam',
       'HP1526901T2D_G17.bam', 'HP1526901T2D_G21.bam', 'HP1526901T2D_G4.bam',
       'HP1526901T2D_H10.bam', 'HP1526901T2D_H20.bam', 'HP1526901T2D_H22.bam',
       'HP1526901T2D_H24.bam', 'HP1526901T2D_H4.bam', 'HP1526901T2D_I13.bam',
       'HP1526901T2D_I3.bam', 'HP1526901T2D_J21.bam', 'HP1526901T2D_K20.bam',
       'HP1526901T2D_K23.bam', 'HP1526901T2D_K3.bam', 'HP1526901T2D_K7.bam',
       'HP1526901T2D_L1.bam', 'HP1526901T2D_L5.bam', 'HP1526901T2D_

In [5]:
df_combined = pd.concat([df_healthy, df_t2d], axis=1, join='inner')

In [6]:
len(df_combined.columns) # Should be 97 + 45 = 142

154

In [16]:
print(df_combined.columns[df_combined.columns.duplicated()])


Index(['Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length'], dtype='object')


In [17]:
metadata_cols = ['Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length']
count_cols = [col for col in df_combined.columns if col not in metadata_cols]


In [19]:
from collections import defaultdict

dupes = df_combined.columns[df_combined.columns.duplicated()].unique()
identical = defaultdict(bool)

for col in dupes:
    cols = df_combined.loc[:, df_combined.columns == col]
    identical[col] = (cols.nunique(axis=1) == 1).all()

print({k: v for k, v in identical.items()})


{'Geneid': True, 'Chr': True, 'Start': True, 'End': True, 'Strand': True, 'Length': True}


In [23]:
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]

In [22]:
print(len(df_combined.columns))

148


In [18]:
non_numeric = df_combined[count_cols].apply(lambda col: ~pd.to_numeric(col, errors='coerce').notnull()).any()
non_numeric_cols = non_numeric[non_numeric].index.tolist()

if non_numeric_cols:
    print("Non-numeric values found in:", non_numeric_cols)
else:
    print("All count columns are numeric.")



All count columns are numeric.


In [21]:
df_combined.to_csv("counts_combined.txt", sep="\t", index=False)